# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guideline for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # mlcroissant's Dataset.metadata is an object
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields. Each entity is referenced by its `@id` according to the Croissant schema.

Let's examine the available record sets, fields and columns. **Note:** Use `@id` to reference any dataset elements.

In [ ]:
# View top-level record sets, their `@id`, and fields
record_sets = dataset.record_sets
print("Available RecordSets (by @id):")
for rs in record_sets:
    print(f"  - RecordSet @id: {rs['@id']}, name: {rs.get('name', '')}")

# Explore the fields of one record set for illustration
if record_sets:
    the_record_set = record_sets[0]
    print(f"\nFields in RecordSet @id '{the_record_set['@id']}':")
    for field in the_record_set.get('field', []):
        # Each field is a dict with '@id' and usually 'name'
        print(f"  * Field @id: {field['@id']} - {field.get('name', '')}")
    # List columns for this record set if present
    if 'column' in the_record_set:
        print(f"\nColumns in RecordSet '{the_record_set['@id']}':")
        for col in the_record_set['column']:
            print(f"    - Column @id: {col['@id']} - {col.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets
# Use the @id to uniquely reference each record set
dataframes = {}
rs_ids = [rs['@id'] for rs in dataset.record_sets]
for record_set_id in rs_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows for record set {record_set_id}")

# Show columns for first available record set
if rs_ids:
    first_rs = rs_ids[0]
    print(f"\nColumns in '{first_rs}':\n", dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming distributions, or grouping data by key attributes.

> ⚠️ **Note:** Please update `<numeric_field_id>` and `<group_field_id>` to match valid `@id`s from your dataset (see previous outputs for the real names). Below demonstrates an example workflow.

In [ ]:
# Example: Filter and analyze a numeric field
# Please replace these with actual valid field names (@id) from your data overview!
# For demonstration, let's attempt to guess likely fields
first_rs_id = rs_ids[0] if rs_ids else None
df = dataframes.get(first_rs_id, pd.DataFrame())
if not df.empty:
    # Try to find a numeric column (example: 'mw' for molecular weight, 'coverage', etc.)
    import numpy as np
    candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if not candidates:
        # Try converting possible numeric fields
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if candidates:
        numeric_field = candidates[0]
        print(f"Using numeric field: {numeric_field}")

        threshold = df[numeric_field].mean()  # Use mean as threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered {len(filtered_df)} records with {numeric_field} > mean ({threshold:.2f})")

        norm_col = numeric_field + "_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"First 5 normalized values for '{numeric_field}':")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Try finding a grouping field (categorical, e.g. 'description' or 'modification')
        group_fields = [col for col in df.columns if df[col].dtype==object and col != numeric_field]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouping by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
    else:
        print("No numeric field found for analysis.")
else:
    print("No data available in the first record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Edit fields as appropriate for your analysis.

In [ ]:
# Example: Histogram and scatter plot for one numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.show()

    # Scatter plot example if more than one numeric
    if len(candidates) >= 2:
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=df[candidates[0]], y=df[candidates[1]])
        plt.xlabel(candidates[0])
        plt.ylabel(candidates[1])
        plt.title(f"Scatter plot: {candidates[0]} vs {candidates[1]}")
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
This notebook guided you through the exploration of the FAIR^2 dataset using `mlcroissant`. We've loaded metadata, examined record sets and fields by `@id`, extracted and processed data using pandas, and created visualizations. For further domain-specific analyses, adapt field selection and EDA steps with knowledge of the dataset's biology and the mass spectrometry domain.